In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import pandas as pd

import time
import random
import re

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

from typing import List, Tuple, Dict

from tqdm import tqdm

In [ ]:
data_path = 'gecodb_v01.tsv'

### Отбор данных [1 балл]

In [ ]:
df = pd.read_csv(data_path, sep='\t', header=None)

In [ ]:
df.columns = ['Word', 'Count']

In [ ]:
df.head()

In [ ]:
df.shape

Для начала отберём из множества слов только те, которые в качестве разделителя имеют ТОЛЬКО нижнее подчёркивание. Также не забываем о существовании букв немецкого алфавита. Регулярные выражения вам в помощь! :)

In [ ]:
data = ...
data.shape

Разумеется, вы можете взять больше слов, всё зависит от возможностей вашего компьютера. В нашем случае мы берём миллион слов.

In [ ]:
data = data.sample(n=1000000)

Теперь посмотрим на количество нижних подчёркиваний в словах. Такая информация нам пригодится для разделения данных на тренировочную и тестовую выборки. Обе выборки должны получиться репрезентативными.

In [ ]:
underscores_counts = ...

In [ ]:
underscores_counts.describe()

In [ ]:
underscores_counts.hist()

In [ ]:
pd.DataFrame({'underscores_counts': underscores_counts}).boxplot()

In [ ]:
underscores_counts.value_counts()

Создадим словари символов и тегов

In [ ]:
chars = set([ch.lower() for word in data['Word'].values for ch in word if ch not in '_+'])
len(chars)

In [ ]:
TAGS = (0, 1, 2)

char2index = {ch: i + 2 for i, ch in enumerate(list(chars))}
char2index['-OOV-'] = 1
char2index['-PAD-'] = 0
char2index[None] = 1

In [ ]:
len(char2index)

### Простая разметка и разбиение данных на обучающую и тестовую выборку [2 балла]

Теперь займёмся предобработкой слов.

Для начала будем считать, что выходная последовательность состоит из двух чисел. Цифрой `1` будем указывать границу подстроки, то есть символ, стоящий перед `_`. В остальных случаях заполняем последовательность нулями.

Итак, заполним позиции символов следующим образом:

`0 - <Word-In>`

`1 - <Word-End>`

In [ ]:
def simple_compound_annotation(word: str) -> str:
    """
    Данная функция кодирует слово в последовательность из 0 и 1
    1 - <Word-End> - буква, являющаяся границей подслова, т.е. стоящая перед `_`
    0 - <Word-In - в противном случае
    :param word: слово
    :return строка со списком
    """
    seq = []
    # YOUR CODE HERE
    return str(seq)

In [ ]:
simple_compound_annotation('Zeit_Punkt')

In [ ]:
data['Annotation'] = data['Word'].apply(simple_compound_annotation)

In [ ]:
data['Underscores'] = ...

In [ ]:
data.head(3)

Перемешаем слова, чтобы они не следовали в алфавитном порядке.

In [ ]:
data = shuffle(data)
data.head(10)

Разделим выборку на тренировочную и тестовую. Обратим внимание, что в распределении больше всего двусложных и трёхсложных слов. Поэтому мы их в первую очередь и возьмём в обучающие данные. А большую часть многосложных слов (особенно если их совсем немного) отправим в тестовую выборку.

На всякий случай, мы не берём в выборку слова, которые не содержат никаких подчёркиваний :)

Также никто не мешает вам создать валидационную выборку и потом корректировать по ней качество модели :)

In [ ]:
train = ...
val = ...
test = ...

In [ ]:
train.shape, test.shape

In [ ]:
train = shuffle(train)
test = shuffle(test)

In [ ]:
train.Underscores.value_counts()

In [ ]:
test.Underscores.value_counts()

In [ ]:
def list2tensor(l):
    return torch.tensor(l, dtype=torch.long)

def encode_words(words: List[str]) -> List[List[float]]:
    """
    Напишите функцию, кодирующую слова в последовательности символьных кодов
    Обратите внимание на то, чтобы функция умела обрабатывать символы, которых нет в словаре.
    Также не забываёте о том, что все нижние подчёркивания и маркеры для инфиксов должны быть удалены из слов.
    Чтобы быстрее обрабатывать слова, рекомендуется вместо питоновских списков использвать функцию np.zeros(...)
    и затем добавлять код символа по индексу

    :param words: список слов
    :return массив с тензорами, состоящими из символьных кодов
    """
    words_X = []
    # YOUR CODE HERE
    return words_X

def encode_tags(tags):
    return [list2tensor(eval(ts)) for ts in tags]

In [ ]:
train_x, train_y = encode_words(train['Word'].values), encode_tags(train['Annotation'].values)

In [ ]:
test_x, test_y = encode_words(test['Word'].values), encode_tags(test['Annotation'].values)

### Разделитель слов [3 балла]

Построим сегментатор на основе символьной языковой модели.

In [ ]:
class BiLSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size, dropout=0.2):
        super(BiLSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.embeddings = ... # строим symbol2vec-матрицу
        self.bilstm = ... # двунаправленный LSTM-слой
        self.dense = ... # обычный fc-layer
        self.dropout = ... # позволим нашей модели не переобучаться на трейне :)

    def forward(self, word):
        """
        Напишите код для архитектуры сети
        В конце вам может пригодиться softmax
        """
        # YOUR CODE HERE
        return tag_scores

### Обучение + валидация [1 балл + 1 балл]

In [ ]:
EMBEDDING_DIM = 32
HIDDEN_DIM = 32
N_EPOCHS = 3
N_TAGS = 2

In [ ]:
model = BiLSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, ...)
loss_function = ...
optimizer = ...

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
@torch.no_grad()
def print_metrics(model, data, batch_size=BATCH_SIZE, name="", **kw):
    model.eval()
    # YOUR CODE HERE
    # Берите любые метрики, которые хотите оценивать на валидационной выборке

In [ ]:
for epoch in range(N_EPOCHS):
    model.train()
    running_loss = 0.0
    for i, (word_in, targets) in enumerate(zip(train_x, train_y)):
        model.zero_grad()
        tag_scores = model(word_in)
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

        # print statistics

        running_loss += loss.item()
        if i % 10000 == 9999:
            print(f'[{epoch + 1}, {i + 1:5d}/{len(train_x)}] loss: {running_loss / 10000:.3f}')
            running_loss = 0.0

    ### здесь может быть оценена ваша валидационная выборка
    print_metrics(model, val_data, max_len=128)

In [ ]:
predicts = ...

In [ ]:
y_true = [l for seq in test_y for l in seq]
y_pred = [l for seq in predicts for l in seq]

In [ ]:
target_names = ('Word-In', 'Word-End')

print(classification_report(y_true, y_pred, target_names=target_names))

### Усложним нашу разметку для инфиксов [1 балл]

Пусть нашему инфиксу будет присваиваться код 2 (`Infix-End`). В итоге мы будем иметь три тега.

In [ ]:
def encode_complex_compound_annotation(word):
    """
    Данная функция кодирует слово в последовательность из 0 и 1
    2 - <Infix-End> - буква, являющаяся границей инфикса, т.е. последняя после `+`
    1 - <Word-End> - буква, являющаяся границей подслова, т.е. стоящая перед `_`
    0 - <Word-In - в противном случае
    :param word: слово
    :return строка со списком
    """
    seq = []
    # YOUR CODE HERE
    return str(seq)

In [ ]:
encode_complex_compound_annotation('Vertrieb_+s_Netz_Planung')

### Обучите ту же модель для новой разметки, при желании используйте валидационную выборку [2 балла + 1 балл]

1. Объединим первую обучающую выборку с подвыборкой, в которой находятся инфиксы. Размер подвыборки с инфиксами возьмите в таком расчёте: `x = (количество_слов_с_инфиксами * N_sample)/(количество_слов_только_с_подчёркиванием)`, где `N_sample` - число слов только с подчёркиванием, которые мы взяли в первой выборке

 - N_sample в нашем случае равен 1000000
 - количество_слов_только_с_подчёркиванием, количество_слов_с_инфиксами - ответ находится в команде shape

2. Запускаем обучение (можно использовать валидационную выборку)
3. Оцениваем качество и описываем выводы

В пункте 1 не забываем разделить выборку на train, val и test :)

In [ ]:
infixes_data['Annotation'] = infixes_data['Word'].apply(encode_complex_compound_annotation)

In [ ]:
full_train = ...
full_val = ...
full_test = ...

Обучение

In [ ]:
EMBEDDING_DIM = 32
HIDDEN_DIM = 32
N_EPOCHS = 3
N_TAGS = 3

In [ ]:
model = BiLSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(char2index), N_TAGS)
loss_function = ...
optimizer = ...

In [ ]:
sum(p.numel() for p in model.parameters())

### Придумайте, как можно улучшить качество на втором типе разметки [2 балла]

Идеи

- добавить в архитектуру больше слоёв
- подумать, можно ли здесь применить attention-механизм или crf-слой
- а может здесь имеет смысл дообучение первой модели на новых данных?
- ваши пожелания :)

In [ ]:
<...>